# **Track Network - Finger Plate Number**

### Data Fetching

In [1]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.94.14.115


C:\Users\win 11\AppData\Local\Temp\ipykernel_17428\2728031397.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 7760 rows from 'extraction'


In [2]:
keywords = ["FingerPlateUpBeam", "FingerPlateDownBeam"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Track-Network') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df.shape
df.head(5)

,filename,workorder_id,json_data
1949,TN_PM_MTH_FingerPlateDownBeam_4000452791.pdf,4.000453e+09,"{'notification': {'notification_no': 'NA', 'no..."
1968,TN_PM_MTH_FingerPlateDownBeam_4000452409.pdf,4.000452e+09,"{'notification': {'notification_no': 'NA', 'no..."
1975,TN_PM_MTH_FingerPlateDownBeam_4000452408.pdf,4.000452e+09,"{'notification': {'notification_no': 'NA', 'no..."
2165,TN_PM_MTH_FingerPlateDownBeam_4000519858.pdf,4.000520e+09,"{'notification': {'notification_no': 'NA', 'no..."
2617,TN_PM_MTH_FingerPlateUpBeam_4000662480.pdf,4.000662e+09,"{'notification': {'notification_no': 'NA', 'no..."


In [3]:
import os
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_like(val):
    """Detect NA-like values."""
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    """Return keys in dict where value is NA-like."""
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]


def clean_value(val):
    """Recursively clean NA-like values in dict, list, string."""
    if isinstance(val, str):
        return '' if pattern_na.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

def extract_leaf_keys(d, parent=''):
    """Extract flattened leaf keys from nested dict."""
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                keys.extend(extract_leaf_keys(v, full_key))
            else:
                keys.append(full_key)
    return keys

In [6]:
import string, re

def normalize_fp_key(key):
    return key.strip().lower()

def flatten_finger_plate_bolt_inspection(data):
    flat = {}

    if not isinstance(data, dict):
        return flat

    # 🔁 normalize legacy keys
    for legacy_key in [
        "finger_plate_up_inspection",
        "finger_plate_up_inspections"
    ]:
        if legacy_key in data:
            data["finger_plate_bolts_inspections"] = data.pop(legacy_key)

    fp_root = data.get("finger_plate_bolts_inspections")
    if not isinstance(fp_root, dict):
        return flat

    # ───────────────────────────────
    # Loop each finger plate
    # ───────────────────────────────
    for plate_key, plate_obj in fp_root.items():
        if not isinstance(plate_obj, dict):
            continue

        prefix = plate_key  # e.g. finger_plate_1

        # 1️⃣ Plate-level metadata
        flat[f"{prefix}.inspection_date"] = plate_obj.get("inspection_date")
        flat[f"{prefix}.finger_plate_no"] = plate_obj.get("finger_plate_no")

        # 2️⃣ Side → Bolt → torque / change
        for k, v in plate_obj.items():

            if not isinstance(v, dict):
                continue

            # skip non-side blocks
            if not k.startswith("side"):
                continue

            for bolt_key, bolt_obj in v.items():
                if not isinstance(bolt_obj, dict):
                    continue

                flat[f"{prefix}.{k}.{bolt_key}.torque"] = bolt_obj.get("torque")
                flat[f"{prefix}.{k}.{bolt_key}.change"] = bolt_obj.get("change")

        # 3️⃣ Plate-level closing fields
        flat[f"{prefix}.faults"] = plate_obj.get("faults")

        tech = plate_obj.get("technician", {})
        sup = plate_obj.get("supervisor", {})

        flat[f"{prefix}.technician.technician_id"] = (
            tech.get("technician_id") if isinstance(tech, dict) else None
        )

        flat[f"{prefix}.supervisor.supervisor_id"] = (
            sup.get("supervisor_id") if isinstance(sup, dict) else None
        )

    return flat

flattened_rows = [
    flatten_finger_plate_bolt_inspection(row)
    for row in df["json_data"]
]

fingerplate_df = pd.DataFrame(flattened_rows)

fingerplate_df.insert(0, "workorder_id", df["workorder_id"].astype("Int64").to_numpy())
fingerplate_df.insert(1, "filename", df["filename"].to_numpy())


rename_map = {}

for col in fingerplate_df.columns:
    new_col = re.sub(
        r"\.(technician|supervisor)\.\1_(id|date)$",
        r".\1_\2",
        col
    )

    if new_col != col:
        rename_map[col] = new_col

fingerplate_df = fingerplate_df.rename(columns=rename_map)

list(fingerplate_df.columns)

['workorder_id',
 'filename',
 'finger_plate_1.inspection_date',
 'finger_plate_1.finger_plate_no',
 'finger_plate_1.side_a1.bolt1.torque',
 'finger_plate_1.side_a1.bolt1.change',
 'finger_plate_1.side_a1.bolt2.torque',
 'finger_plate_1.side_a1.bolt2.change',
 'finger_plate_1.side_a1.bolt3.torque',
 'finger_plate_1.side_a1.bolt3.change',
 'finger_plate_1.side_a1.bolt4.torque',
 'finger_plate_1.side_a1.bolt4.change',
 'finger_plate_1.side_a1.bolt5.torque',
 'finger_plate_1.side_a1.bolt5.change',
 'finger_plate_1.side_a1.bolt6.torque',
 'finger_plate_1.side_a1.bolt6.change',
 'finger_plate_1.side_a1.bolt7.torque',
 'finger_plate_1.side_a1.bolt7.change',
 'finger_plate_1.side_a1.bolt8.torque',
 'finger_plate_1.side_a1.bolt8.change',
 'finger_plate_1.side_a1.bolt9.torque',
 'finger_plate_1.side_a1.bolt9.change',
 'finger_plate_1.side_a1.bolt10.torque',
 'finger_plate_1.side_a1.bolt10.change',
 'finger_plate_1.side_a2.bolt1.torque',
 'finger_plate_1.side_a2.bolt1.change',
 'finger_plate_1.s

In [18]:
import re

def column_sort_key(col):
    # Always keep these first
    if col == "workorder_id":
        return (0, 0, 0, 0, 0, 0)
    if col == "filename":
        return (0, 0, 0, 0, 0, 1)

    # finger_plate_X.xxx
    m = re.match(r"finger_plate_(\d+)\.(.*)", col)
    if not m:
        return (99, 99, 99, 99, 99, col)

    plate_no = int(m.group(1))
    rest = m.group(2)

    # ---- section priority ----
    if rest.startswith("side"):
        section_priority = 0
    elif rest.startswith("technician"):
        section_priority = 1
    elif rest.startswith("supervisor"):
        section_priority = 2
    elif rest.startswith("comment"):
        section_priority = 3
    else:
        section_priority = 4

    # ---- extract side + bolt numbers ----
    side_letter = ""
    side_no = 0
    bolt_no = 0

    side_match = re.search(r"side_([a-z])(\d+)", rest)
    if side_match:
        side_letter = side_match.group(1)
        side_no = int(side_match.group(2))

    bolt_match = re.search(r"bolt(\d+)", rest)
    if bolt_match:
        bolt_no = int(bolt_match.group(1))

    # ---- bolt field priority ----
    if rest.endswith(".torque"):
        field_priority = 0
    elif rest.endswith(".change"):
        field_priority = 1
    else:
        field_priority = 2

    return (
        1,                 # after workorder_id & filename
        plate_no,          # finger plate number
        section_priority,  # side / technician / supervisor
        side_letter,       # side_a before side_b
        side_no,           # side_a1 before side_a2
        bolt_no,           # bolt1, bolt2, ..., bolt10
        field_priority,    # torque before change
        rest               # final stable fallback
    )

ordered_cols = sorted(fingerplate_df.columns, key=column_sort_key)
fingerplate_df = fingerplate_df[ordered_cols]

fingerplate_df = fingerplate_df.drop(
    columns=[
        col for col in fingerplate_df.columns
        if col.startswith(("side_", "technician", "supervisor"))
    ]
)

In [19]:
for i in fingerplate_df:
    print(i)

workorder_id
filename
finger_plate_1.side_a1.bolt1.torque
finger_plate_1.side_a1.bolt1.change
finger_plate_1.side_a1.bolt2.torque
finger_plate_1.side_a1.bolt2.change
finger_plate_1.side_a1.bolt3.torque
finger_plate_1.side_a1.bolt3.change
finger_plate_1.side_a1.bolt4.torque
finger_plate_1.side_a1.bolt4.change
finger_plate_1.side_a1.bolt5.torque
finger_plate_1.side_a1.bolt5.change
finger_plate_1.side_a1.bolt6.torque
finger_plate_1.side_a1.bolt6.change
finger_plate_1.side_a1.bolt7.torque
finger_plate_1.side_a1.bolt7.change
finger_plate_1.side_a1.bolt8.torque
finger_plate_1.side_a1.bolt8.change
finger_plate_1.side_a1.bolt9.torque
finger_plate_1.side_a1.bolt9.change
finger_plate_1.side_a1.bolt10.torque
finger_plate_1.side_a1.bolt10.change
finger_plate_1.side_a2.bolt1.torque
finger_plate_1.side_a2.bolt1.change
finger_plate_1.side_a2.bolt2.torque
finger_plate_1.side_a2.bolt2.change
finger_plate_1.side_a2.bolt3.torque
finger_plate_1.side_a2.bolt3.change
finger_plate_1.side_a2.bolt4.torque
fing

### Output Excel

In [20]:
import os
import pandas as pd
from openpyxl import Workbook

output_path = '../../output/tnm/finger_plate.xlsx'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

if not os.path.exists(output_path):
    Workbook().save(output_path)

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    fingerplate_df.to_excel(writer, index=False, sheet_name='finger_plate'),

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")


✅ Exported successfully to '../../output/tnm/finger_plate.xlsx' (replaced existing sheet)
